# Great Expectations: property transaction quality checks

This notebook reads the dbt `fact_transaction` table from BigQuery and validates it with a Great Expectations pandas batch. It checks transaction key integrity, positive measures when present, and sale-date validity. It does not modify warehouse data.

## Prerequisites

Install the dependencies in the active notebook environment:

```bash
pip install great_expectations pandas google-cloud-bigquery
```

Authenticate outside the notebook with Google Application Default Credentials (for example, `gcloud auth application-default login`) and set `BIGQUERY_PROJECT` to the Google Cloud project containing the dbt output. Optionally set `BIGQUERY_DATASET` (defaults to `propery_trans1`). No credentials are stored in this notebook.

## GX workflow mapping

This notebook follows the GX flow shown in the training diagram:

1. Data Context
2. Data Source
3. Data Asset
4. Batch
5. Expectations
6. Expectation Suite
7. Validation

The code below creates the context, attaches the BigQuery table as a pandas batch, defines expectations, bundles them into a suite, and runs validation.

In [1]:
import os
import re

import great_expectations as gx
import pandas as pd
from google.cloud import bigquery

# Default to the project used by the Meltano BigQuery loader in this workspace.
# Override BIGQUERY_PROJECT, BIGQUERY_DATASET, and BIGQUERY_TABLE in the environment
# if you need to point the validation at another warehouse target.
project_id = os.environ.get("BIGQUERY_PROJECT", "ringed-griffin-509403-b4")
dataset_id = os.environ.get("BIGQUERY_DATASET", "propery_trans1")
table_id = os.environ.get("BIGQUERY_TABLE", "fact_transaction")

if not project_id:
    raise EnvironmentError("Set BIGQUERY_PROJECT to the GCP project containing the dbt tables.")

for label, value in (("BIGQUERY_PROJECT", project_id), ("BIGQUERY_DATASET", dataset_id), ("BIGQUERY_TABLE", table_id)):
    if not re.fullmatch(r"[A-Za-z0-9_-]+", value):
        raise ValueError(f"{label} contains unsupported characters: {value!r}")

table_ref = f"`{project_id}.{dataset_id}.{table_id}`"
client = bigquery.Client(project=project_id)

query_result = client.query(f"SELECT * FROM {table_ref}").result()
column_names = [field.name for field in query_result.schema]
transactions = pd.DataFrame.from_records(
    (tuple(row) for row in query_result),
    columns=column_names,
)
print(f"Loaded {len(transactions):,} rows from {project_id}.{dataset_id}.{table_id}")
transactions.head()

Loaded 170,224 rows from ringed-griffin-509403-b4.propery_trans1.fact_transaction


,transaction_key,project_key,location_key,sale_date,transacted_price,nett_price,area_sqft,area_sqm,unit_price_psf,new_trans_price,percentage_difference,unit_price_psm,number_of_units,type_of_sale,type_of_area,purchaser_address_indicator
0,3474790414281159860,-4852532521223715081,6021146233820805257,2020-11-06,8000000,None,5048.32,469,1585,8001587,0.000198375,17058,1,RESALE,STRATA,PRIVATE
1,5619611621151717454,754663682913641818,-5508014609348575077,2016-03-10,680000,None,1119.46,104,607,679512,-0.000717647,6538,1,RESALE,STRATA,PRIVATE
2,2270959200694403351,6030734890887807752,5114548285489408306,2021-11-15,700000,None,1119.46,104,625,699663,-0.000482857,6731,1,RESALE,STRATA,HDB
3,911406479674455196,754663682913641818,7947363692064715591,2015-05-22,730000,None,1119.46,104,652,729888,-0.000153425,7019,1,RESALE,STRATA,PRIVATE
4,6204921242229294112,754663682913641818,-249969701356554278,2013-08-02,735000,None,1119.46,104,657,735485,0.000659864,7067,1,RESALE,STRATA,HDB


## Define and run the validation suite

Nullable measure values are allowed because the dbt fact model cleans nonpositive values to `NULL`. Any measure that is present must be greater than zero. Date values are parsed strictly; a non-null value that cannot be parsed fails validation.

## Step 5: define expectations

The validation rules are stored in a suite so they can be reviewed and reused. Each expectation checks a specific data-quality rule for the transaction fact table.

## Step 6: bundle the expectations into a suite

The `ExpectationSuite` is registered in the current GX context and the validation definition binds it to the batch.

## Step 7: run validation

This final step executes the suite against the chosen batch and surfaces any failing expectations.

In [2]:
from duckdb import df


required_columns = {
    "transaction_key",
    "transacted_price",
    "nett_price",
    "area_sqft",
    "area_sqm",
    "unit_price_psf",
    "unit_price_psm",
    "number_of_units",
    "sale_date",
}
missing_columns = sorted(required_columns.difference(transactions.columns))
if missing_columns:
    raise ValueError(f"fact_transaction is missing expected columns: {missing_columns}")

# Preserve null dates, but convert malformed non-null dates to an invalid sentinel for validation.
transactions = transactions.copy()
original_sale_date = transactions["sale_date"]
parsed_sale_date = pd.to_datetime(original_sale_date, errors="coerce")
invalid_sale_date = original_sale_date.notna() & parsed_sale_date.isna()
transactions["sale_date_parse_valid"] = ~invalid_sale_date

# Step 1: create the GX data context
# Step 2: define a pandas data source
# Step 3: add the fact_transaction dataframe as a data asset
# Step 4: create a whole-dataframe batch
context = gx.get_context(mode="ephemeral")
data_source = context.data_sources.add_pandas(name="property_transactions_pandas")
data_asset = data_source.add_dataframe_asset(name="fact_transaction")
batch_definition = data_asset.add_batch_definition_whole_dataframe(name="fact_transaction_batch")

suite = gx.ExpectationSuite(name="fact_transaction_quality")
suite.add_expectation(gx.expectations.ExpectColumnToExist(column="transaction_key"))
suite.add_expectation(gx.expectations.ExpectColumnValuesToNotBeNull(column="transaction_key"))
suite.add_expectation(gx.expectations.ExpectColumnValuesToBeUnique(column="transaction_key"))

suite.add_expectation(gx.expectations.ExpectColumnValuesToNotBeNull(column="sale_date"))

positive_measure_columns = [
    "transacted_price",
    "nett_price",
    "area_sqft",
    "area_sqm",
    "unit_price_psf",
    "unit_price_psm",
    "new_trans_price",
    "number_of_units",
]
for column in positive_measure_columns:
    suite.add_expectation(
        gx.expectations.ExpectColumnValuesToBeBetween(
            column=column,
            min_value=0,
            strict_min=True,
            mostly=1.0,
        )
    )

suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeInSet(
        column="sale_date_parse_valid",
        value_set=[True],
    )
)

suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeBetween(column="transacted_price", min_value=0, max_value=75000000, strict_min=True, strict_max=True)
)

suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeBetween(column="percentage_difference", min_value=-0.005, max_value=0.005, strict_min=True, strict_max=True)
)

suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeBetween(
        column="new_trans_price",
        min_value=0,
        strict_min=True,
        mostly=1.0,
    )
)


suite.add_expectation(
    gx.expectations.ExpectColumnPairValuesToBeEqual(
        column_A="transacted_price",
        column_B="new_trans_price",
        ignore_row_if="both_values_are_missing"
    )
)

suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeBetween(column="number_of_units", min_value=1, max_value=1)
)


context.suites.add(suite)
validation_definition = gx.ValidationDefinition(
    name="validate_fact_transaction",
    data=batch_definition,
    suite=suite,
)
context.validation_definitions.add(validation_definition)
validation_results = validation_definition.run(
    batch_parameters={"dataframe": transactions}
)

print(f"Overall validation: {'PASS' if validation_results.success else 'FAIL'}")
print(f"Successful expectations: {validation_results.statistics['successful_expectations']}")
print(f"Unsuccessful expectations: {validation_results.statistics['unsuccessful_expectations']}")

failed_results = [
    result for result in validation_results.results if not result.success
]
for result in failed_results:
    print(
        f"FAILED: {result.expectation_config.type} "
        f"{result.expectation_config.kwargs} -> {result.result}"
    )


#if not validation_results.success:
    #raise AssertionError("Great Expectations validation failed; inspect the failed expectations above.")

Calculating Metrics: 100%|██████████| 107/107 [00:02<00:00, 41.42it/s]


Overall validation: FAIL
Successful expectations: 13
Unsuccessful expectations: 4
FAILED: expect_column_values_to_be_between {'batch_id': 'property_transactions_pandas-fact_transaction', 'column': 'transacted_price', 'min_value': 0.0, 'max_value': 75000000.0, 'strict_min': True, 'strict_max': True} -> {'element_count': 170224, 'unexpected_count': np.int64(73), 'unexpected_percent': np.float64(0.04288466961180562), 'partial_unexpected_list': [Decimal('629000000'), Decimal('136000000'), Decimal('550800000'), Decimal('92714566'), Decimal('575000000'), Decimal('180650000'), Decimal('477940000'), Decimal('334200000'), Decimal('375497670'), Decimal('86499999'), Decimal('261644000'), Decimal('610000000'), Decimal('610000000'), Decimal('201080000'), Decimal('95000000'), Decimal('95000000'), Decimal('168000000'), Decimal('168000000'), Decimal('95970784'), Decimal('499000000')], 'missing_count': np.int64(0), 'missing_percent': np.float64(0.0), 'unexpected_percent_total': np.float64(0.04288466961